# XGBoost 入門 — 決定木から XGBoost へ

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kyorin-phys/MLIntro2026/blob/main/appendix/xgboost_intro.ipynb)

**XGBoost** は、決定木をたくさん組み合わせて予測する手法 (勾配ブースティング) のライブラリです。
検査値が並んだ **表形式のデータ** の分類・予測に強く、データ分析コンペでもよく使われます。

このノートブックでは、第3回で使った **乳癌の診断データ** を使い、XGBoost の土台である **決定木** と、その発展形の **ランダムフォレスト** から順に見ていきます。

| 章 | 内容 |
|---|---|
| 1〜3 | 準備、データ、学習用とテスト用に分ける |
| 4〜6 | **決定木 → ランダムフォレスト → XGBoost** の順に分類器を作る |
| 7 | 交差検証で、ロジスティック回帰 (第3回) も含めた4つの手法を比べる |
| 8 | **XGBoost が力を発揮する場面** (U字型・組み合わせ・データの多さ) |
| 9 | **落とし穴：同じ患者のデータが複数行あるとき** |
| 10 | どの特徴量が効いているかを見る (特徴量の重要度、SHAP) |
| 11〜12 | 設定を変える、自分のデータで試す |

XGBoost の入力は **表** です。画像そのものから判定したい場合は、CNN などの別の手法になります。
画像から数値 (特徴量) を取り出してから XGBoost に渡す方法もあります (PyRadiomics の教材を参照)。

> **このノートブックの範囲**：機械学習の分類器の作り方と評価の仕方を学ぶ教育用の入門です。医学的な解釈は扱いません。
> 注意点は最後の「範囲と注意点」にまとめています。

> **この教材の作り方**：コードと説明文は、すべて生成 AI (Claude) との対話で作成しました。作成者はコードを1行も書いていません。
> 作成者が行ったのは、作りたいものを言葉で伝えること、実行して動作を確かめること、分かりにくい点や問題点を指摘して直させることです。

## 1. 準備

このノートブックは **Google Colab** で実行します。XGBoost と SHAP をインストールし、使うライブラリを読み込みます。

In [ ]:
!pip install -q xgboost shap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb

print('xgboost', xgb.__version__)

## 2. データ

第3回と同じ、乳癌の診断データ (569例) です。細胞核の画像から測った30個の値 (半径・面積・凹み具合など) から、悪性か良性かを予測します。

第3回と同じく、**悪性を 1 (陽性)、良性を 0 (陰性)** に変換します。

In [ ]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer(as_frame=True)
X = cancer.data                              # 30個の特徴量 (列名つきの表)
y = (cancer.target == 0).astype(int)         # 元は 0=悪性 なので、悪性を1に変換

print('症例数:', len(X), ' 特徴量の数:', X.shape[1])
print('悪性の割合:', round(y.mean(), 3))
X.head()

## 3. 学習用とテスト用に分ける

第3回と同じ分け方 (テスト用3割、`random_state=0`) にします。学習用でモデルを学習させ、学習に使っていないテスト用で性能を確かめます。

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
print('学習用:', len(X_train), '例  テスト用:', len(X_test), '例')

## 4. 決定木

**決定木** は、「面積は 950 より大きいか？」のような **はい/いいえの質問** を繰り返してデータを分けていく手法です。
フローチャートと同じ形なので、何を根拠に判定したかが一目で分かります。

どの特徴量のどの値で分けるかは、悪性/良性が一番きれいに分かれるように自動で決まります。
まず深さ3 (質問3段) の木を作ります。

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, roc_auc_score

tree = DecisionTreeClassifier(max_depth=3, random_state=0)
tree.fit(X_train, y_train)                   # fit で学習する

print('テスト用の正解率:', round(accuracy_score(y_test, tree.predict(X_test)), 3))
print('テスト用の AUC   :', round(roc_auc_score(y_test, tree.predict_proba(X_test)[:, 1]), 3))

木を図にします。各箱の見方は次のとおりです。

- 1行目：質問 (例 `worst concave points <= 0.142`)。はい (True) なら左、いいえ (False) なら右へ進む
- `gini`：箱の中の混ざり具合。0 なら全部同じラベル、0.5 に近いほど半々に混ざっている
- `samples`：その箱に来た症例の数
- `value`：[良性, 悪性] の数
- `class`：その箱で多い方 (一番下の箱に来たら、これが予測になる)
- 色：良性と悪性のどちらが多いかで色が分かれ、色が濃いほどはっきりしている

In [ ]:
plt.figure(figsize=(18, 7))
plot_tree(tree, feature_names=list(X.columns), class_names=['benign', 'malignant'],
          filled=True, rounded=True, fontsize=9)
plt.show()

### 深くしすぎると「丸暗記」になる

木を深くするほど、学習用データにぴったり合わせられます。深さの制限をなくすと、学習用の正解率は 1.0 (全問正解) になりますが、テスト用の正解率はかえって下がります。
学習用データの偶然のばらつきまで覚えてしまうためで、これを **過学習** と言います。

In [ ]:
for depth in [1, 2, 3, 5, None]:                 # None = 制限なし
    t = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_train, y_train)
    print(f'max_depth={str(depth):4s}  学習用の正解率 {t.score(X_train, y_train):.3f}   テスト用の正解率 {t.score(X_test, y_test):.3f}')

1本の木は分かりやすい反面、性能に限界があり、データが少し変わるだけで形が大きく変わる (不安定な) 弱点があります。
そこで、**木をたくさん作って組み合わせる** 方法が考えられました。組み合わせ方の違いで、次の2つの手法があります。

| 手法 | 木の作り方 | 予測の仕方 |
|---|---|---|
| ランダムフォレスト | 少しずつ違うデータ・違う特徴量で、木を **それぞれ独立に** たくさん作る | 全部の木の多数決 (平均) |
| XGBoost (勾配ブースティング) | 前の木が **間違えたところを直すように**、木を1本ずつ順に足していく | 全部の木の答えの合計 |

## 5. ランダムフォレスト

**ランダムフォレスト** は、次のようにして少しずつ違う木をたくさん作り、多数決で予測します。

- 木ごとに、学習用データから **ランダムに選び直したデータ** を使う
- 質問を作るたびに、**ランダムに選んだ一部の特徴量** の中から選ぶ

1本1本の木は不正確でも、間違え方がばらばらなので、多数決をとると打ち消し合って安定します。

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=300, random_state=0)   # 木を300本作る
rf.fit(X_train, y_train)

print('テスト用の正解率:', round(accuracy_score(y_test, rf.predict(X_test)), 3))
print('テスト用の AUC   :', round(roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]), 3))

## 6. XGBoost

**XGBoost** は、木を1本ずつ順に足していく手法 (勾配ブースティング) です。

1. まず浅い木を1本作って予測する
2. その予測が **外れた分** をうまく当てるように、次の木を作って足す
3. これを何百回も繰り返す

ランダムフォレストが「大勢が独立に答えて多数決」なら、XGBoost は「前の人の間違いを次の人が直していく」イメージです。
**欠測 (空欄) があってもそのまま学習できる** のも便利な点です (このデータには欠測はありません)。

使い方は決定木・ランダムフォレストと同じで、`fit` で学習し、`predict` / `predict_proba` で予測します。

In [ ]:
model = xgb.XGBClassifier(random_state=0)
model.fit(X_train, y_train)

pred = model.predict(X_test)                 # 0 か 1 の予測
prob = model.predict_proba(X_test)[:, 1]     # 悪性である確率

print('テスト用の正解率:', round(accuracy_score(y_test, pred), 3))
print('テスト用の AUC   :', round(roc_auc_score(y_test, prob), 3))

### 混同行列と ROC 曲線

第3回のロジスティック回帰と同じように、混同行列と ROC 曲線で確かめます。

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=['benign', 'malignant'], ax=ax[0])
RocCurveDisplay.from_predictions(y_test, prob, ax=ax[1])
ax[1].plot([0, 1], [0, 1], 'k--', lw=1)    # でたらめに判定した場合
plt.tight_layout()
plt.show()

## 7. 交差検証で比べる

4〜6章の結果は、データの分け方1通りだけの結果です。分け方が変わると性能も変わるので、これだけで手法の優劣は決められません。
**交差検証** では、データを5つに分け、「4つで学習・1つでテスト」を5回繰り返して平均します。
`StratifiedKFold` は、5つのグループの **悪性の割合をそろえる** 分け方です。

第3回の **ロジスティック回帰** も加えて比べます。値の大きさがばらばらなので、そろえてから (`StandardScaler`) 学習します。

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

models = {
    'Decision tree (depth 3)': DecisionTreeClassifier(max_depth=3, random_state=0),
    'Random forest': RandomForestClassifier(n_estimators=300, random_state=0),
    'XGBoost': xgb.XGBClassifier(random_state=0),
    'Logistic regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=10000)),
}
for name, m in models.items():
    auc = cross_val_score(m, X, y, cv=cv, scoring='roc_auc')
    print(f'{name:24s}: AUC {auc.mean():.3f} ± {auc.std():.3f}')

決定木1本よりも、木をたくさん組み合わせたランダムフォレストや XGBoost の方が良くなります。

一方で、**ロジスティック回帰も同じくらいの性能** です。このデータは比較的易しく、どの手法でもほぼ上限に近いためです。
XGBoost は「いつも一番良い」手法ではありません。データが数百件程度のときは、単純な手法で十分なこともよくあります。
**まず単純な手法で基準を作り、複雑な手法がそれを上回るかを確かめる** のが良い進め方です。

では、XGBoost はどんなときに役に立つのでしょうか。次の章で見ます。

## 8. XGBoost が力を発揮する場面

ロジスティック回帰は、「値が大きいほどリスクが上がる (または下がる)」という **一方向の関係** しか表せません。
実際の医療データには、そうでない関係がよくあります。

| 関係の形 | 例 |
|---|---|
| **U字型** | BMI は低すぎても高すぎてもリスクが上がる |
| **しきい値** | ある検査値が基準値を超えると急にリスクが上がる |
| **組み合わせ** | 「高齢 **かつ** 腎機能が低い」ときだけリスクが上がる |

決定木は「質問で分ける」手法なので、こうした関係を自然に表せます。木をたくさん組み合わせる XGBoost は、それを細かく正確に表せます。

これを確かめるために、**答え (リスクの決まり方) が分かっている人工データ** を作ります。

| 特徴量 | リスクとの関係 (このデータの決まり) |
|---|---|
| `age`, `egfr` (腎機能) | 70歳を超え、**かつ** eGFR が 45 未満だとリスクが大きく上がる |
| `bmi` | 23 から離れるほど (低くても高くても) リスクが上がる (U字型) |
| `crp` (炎症) | 1.5 を超えるとリスクが上がる (しきい値) |
| `sbp` (血圧) | 高いほど少しずつリスクが上がる (一方向) |

※ 数値の大きさや関係は、仕組みを見せるために **作った設定** です。実際の病気の性質を表したものではありません。

In [ ]:
def make_risk_data(n, seed=0):
    rng = np.random.default_rng(seed)
    age = rng.uniform(30, 90, n).round()
    egfr = np.clip(rng.normal(70, 20, n) - 0.4 * (age - 60), 10, 120).round(1)   # 腎機能 (年齢とともに下がる)
    bmi = rng.normal(23, 4, n).round(1)
    sbp = rng.normal(130, 18, n).round()                                         # 収縮期血圧
    crp = rng.lognormal(-1, 1, n).round(2)                                       # 炎症の指標

    logit = (-3.5
             + 4.0 * ((age > 70) & (egfr < 45))     # 組み合わせ：高齢 かつ 腎機能が低い
             + 1.5 * ((bmi - 23) / 4) ** 2          # U字型：BMI は低すぎても高すぎてもリスク
             + 3.0 * (crp > 1.5)                    # しきい値：CRP が 1.5 を超えるとリスク
             + 0.02 * (sbp - 130))                  # 一直線：血圧が高いほど少しずつリスク
    disease = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)

    return pd.DataFrame(dict(age=age, egfr=egfr, bmi=bmi, sbp=sbp, crp=crp, disease=disease))

test = make_risk_data(20000, seed=99)     # 評価用の「新しい患者」 (たくさん用意して評価を安定させる)
print('評価用:', len(test), '人  病気の割合:', round(test['disease'].mean(), 3))
test.head()

学習用データの人数を 200人・1000人・5000人 と増やしながら、4つの手法を学習させ、評価用データの AUC を比べます (少し時間がかかります)。

In [ ]:
X_new, y_new = test.drop(columns='disease'), test['disease']

rows = []
for n in [200, 1000, 5000]:
    train = make_risk_data(n, seed=1)
    X_tr, y_tr = train.drop(columns='disease'), train['disease']
    models = {
        'Decision tree (depth 3)': DecisionTreeClassifier(max_depth=3, random_state=0),
        'Random forest': RandomForestClassifier(n_estimators=300, random_state=0),
        'XGBoost': xgb.XGBClassifier(random_state=0),
        'Logistic regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    }
    for name, m in models.items():
        m.fit(X_tr, y_tr)
        rows.append({'n_train': n, 'model': name, 'AUC': roc_auc_score(y_new, m.predict_proba(X_new)[:, 1])})

pd.DataFrame(rows).pivot(index='n_train', columns='model', values='AUC').round(3)

乳癌データのときとは違い、はっきり差がつきます。

- **ロジスティック回帰** は、データを増やしても AUC がほとんど上がらない (一方向の関係しか表せないので、U字型や組み合わせを取りこぼす)
- **ランダムフォレスト・XGBoost** は、データが増えるほど良くなる
- **決定木1本** は、深さ3では複雑な関係を表しきれない

### なぜ差がつくか — 部分依存プロット

**部分依存プロット** は、ある特徴量だけを動かしたとき、モデルの予測 (病気の確率) が平均してどう変わるかを描いた図です。
モデルが特徴量とリスクの関係を **どんな形で捉えたか** が分かります。5000人で学習した XGBoost とロジスティック回帰を重ねて描きます。

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

train = make_risk_data(5000, seed=1)
X_tr, y_tr = train.drop(columns='disease'), train['disease']
xgb_model = xgb.XGBClassifier(random_state=0).fit(X_tr, y_tr)
lr_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr, y_tr)

features = ['age', 'egfr', 'bmi', 'crp', 'sbp']
fig, ax = plt.subplots(1, len(features), figsize=(16, 3.5), sharey=True)
PartialDependenceDisplay.from_estimator(xgb_model, X_tr, features, response_method='predict_proba', method='brute',
                                        ax=ax, line_kw={'label': 'XGBoost'})
PartialDependenceDisplay.from_estimator(lr_model, X_tr, features, response_method='predict_proba', method='brute',
                                        ax=ax, line_kw={'label': 'Logistic regression', 'color': 'tab:red'})
ax[0].set_ylim(0, 1)     # 縦軸 (確率) を 0〜1 にそろえる
plt.tight_layout()
plt.show()

図の読み方 (縦軸は予測された病気の確率)

- `bmi`：XGBoost は **U字型** を捉えている。ロジスティック回帰はほぼ **水平** で、「BMI はリスクと関係ない」と判断してしまっている
- `crp`：XGBoost は 1.5 付近で **階段状に** 上がる。ロジスティック回帰はなだらかな曲線
- `egfr`：XGBoost は 45 付近で段差がある (高齢者との組み合わせの効果)
- `sbp`：一方向の弱い関係なので、どちらも似た形

ロジスティック回帰でも、`bmi` の2乗の列を自分で加えれば U字型を表せます。ただしそれには **関係の形を前もって知っている** 必要があります。
XGBoost は形を知らなくても、データから自動で見つけてくれます。これが XGBoost の一番の利点です。

逆に言えば、実際のデータで XGBoost とロジスティック回帰の差が小さいときは、「一方向の単純な関係で説明できるデータだった」ということです (7章の乳癌データ)。

## 9. 落とし穴：同じ患者のデータが複数行あるとき

乳癌のデータは **1行 = 1人** でした。しかし実際の医療データでは、**同じ人のデータが複数行** になることがよくあります。

- 両眼、左右の腎臓・乳房など、**左右がある臓器**
- 同じ人の **複数回の受診・検査** (経過観察)
- 1人から取った **複数の病変・複数の画像**

このとき、行単位でランダムに分けると、同じ人のデータが学習用とテスト用の両方に入ります。
同じ人のデータは検査値もラベルもよく似ているので、モデルは「この人は病気だった」ことを覚えて当てられてしまいます。
これは本番 (新しい患者) では使えない手がかりなので、**性能が実際より良く見えてしまいます**。

人工データで確かめます。200人から2行ずつ (合計400行) のデータで、同じ人の2行は検査値の個人差を共有しています。

In [ ]:
def make_grouped_data(n_patients=200, n_rows=2, seed=0):
    rng = np.random.default_rng(seed)
    rows = []
    for pid in range(n_patients):
        base = rng.normal(0, 1, 5)          # 検査値の個人差 (同じ人の行どうしで似る)
        risk_p = rng.normal(0, 1)           # 病気へのなりやすさ (データには入らない)
        age = int(rng.integers(30, 86))
        for r in range(n_rows):
            disease = int(-0.6 + 0.02 * (age - 60) + 2.0 * risk_p + rng.normal(0, 0.5) > 0)
            x = base + rng.normal(0, 0.35, 5) + 0.6 * disease
            rows.append(dict(patient_id=pid, age=age, **{f'test{k + 1}': round(x[k], 2) for k in range(5)},
                             disease=disease))
    return pd.DataFrame(rows)

df = make_grouped_data()
df.head(6)

**行単位** で分けた場合 (`StratifiedKFold`) と、**患者単位** で分けた場合 (`StratifiedGroupKFold` に `groups=患者ID` を渡す) を比べます。

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

Xg = df.drop(columns=['patient_id', 'disease'])
yg = df['disease']
groups = df['patient_id']

cv_group = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)

auc_row = cross_val_score(xgb.XGBClassifier(random_state=0), Xg, yg, cv=cv, scoring='roc_auc')
auc_patient = cross_val_score(xgb.XGBClassifier(random_state=0), Xg, yg, groups=groups, cv=cv_group, scoring='roc_auc')

print(f'行単位で分けた   (StratifiedKFold)     : AUC {auc_row.mean():.3f}   ← 甘い評価')
print(f'患者単位で分けた (StratifiedGroupKFold): AUC {auc_patient.mean():.3f}   ← 正しい評価')

行単位で分けた方が AUC が高く出るはずです。この差は **見かけだけ** の性能です。

対策は次のどちらかです。

| 対策 | 方法 | 注意 |
|---|---|---|
| 患者単位で分ける | `GroupKFold` / `StratifiedGroupKFold` / `GroupShuffleSplit` に `groups=患者ID` を渡す | 全部のデータを使える |
| 1人1行にする | 片側だけ、初回の受診だけ、などに絞る | データが減る |

「**同じ人のデータが学習用とテスト用の両方に入っていないか**」を常に確認しましょう。

## 10. どの特徴量が効いているか

乳癌のデータに戻ります。決定木1本なら図を見れば根拠が分かりますが、何百本もの木を組み合わせたランダムフォレストや XGBoost は、そのままでは中身が分かりません。
そこで、次のような方法で手がかりを調べます。

### 特徴量の重要度

XGBoost は、学習の中でどの特徴量をよく使ったかを数値で出せます。全データで学習し直し、上位15個を見てみます。

In [ ]:
model_all = xgb.XGBClassifier(random_state=0).fit(X, y)

imp = pd.Series(model_all.feature_importances_, index=X.columns).sort_values()
imp.tail(15).plot.barh(figsize=(6, 5))
plt.xlabel('importance')
plt.title('XGBoost feature importance (top 15)')
plt.tight_layout()
plt.show()

### SHAP (任意)

重要度は「どれだけ使ったか」しか分かりません。**SHAP** を使うと、各特徴量が **どちら向きに** (悪性の確率を上げる/下げる) 効いているかが分かります。

図の読み方
- 1つの点が1つの症例
- 横軸：その特徴量が予測をどれだけ「悪性」側 (右) / 「良性」側 (左) に動かしたか
- 色：特徴量の値の大小 (赤=大きい、青=小さい)

例えば `worst area` で「赤い点 (大きい) が右側」に並んでいれば、「面積が大きいほど悪性と予測する」ことを意味します。

In [ ]:
import shap

explainer = shap.TreeExplainer(model_all)
shap_values = explainer.shap_values(X)
shap.summary_plot(shap_values, X, max_display=15)

⚠ 重要度や SHAP は「**このモデルが** 何を手がかりにしたか」を示すもので、「その値が病気の **原因** である」ことを示すものではありません。

## 11. 設定 (ハイパーパラメータ) を変える

XGBoost の主な設定です。

| 設定 | 意味 | 既定値 |
|---|---|---|
| `n_estimators` | 決定木の数 | 100 |
| `max_depth` | 1本の木の深さ (大きいほど複雑なパターンを覚える) | 6 |
| `learning_rate` | 1本の木の影響の大きさ (小さいほど慎重、木の数を増やす必要がある) | 0.3 |
| `subsample` | 1本の木を作るときに使うデータの割合 | 1.0 |
| `scale_pos_weight` | 陽性 (ラベル1) を重視する度合い。陽性が少ないときに使う | 1 |

木の深さ `max_depth` を変えて比べてみます。4章の決定木では深くすると過学習しましたが、XGBoost ではどうでしょうか。

In [ ]:
for depth in [1, 2, 3, 6]:
    m = xgb.XGBClassifier(max_depth=depth, random_state=0)
    auc = cross_val_score(m, X, y, cv=cv, scoring='roc_auc')
    print(f'max_depth={depth}: AUC {auc.mean():.3f}')

このデータでは、深さを変えてもほとんど差がありません。XGBoost は1本1本の木の影響を小さくして足していくので、決定木1本ほど過学習しにくいためです。
データによっては差が出るので、自分のデータでは試してみる価値があります。

⚠ 同じデータで設定を何度も試して一番良いものを選ぶと、その AUC も少し甘くなります。
本格的には、設定選びに使わない **テスト用データを最後まで取っておく** か、入れ子の交差検証 (nested CV) を使います。

## 12. 自分のデータで試す (任意)

Excel (`.xlsx`) や CSV (`.csv`) の表を、Colab の左のファイル欄 (フォルダのアイコン) からアップロードし、ファイル名と列名を書き換えて実行してください。

表は次の形にしておきます。
- **1行 = 1症例**
- ラベル (予測したいもの) の列は 0 / 1
- 文字の列 (性別など) は 0 / 1 などの数値にしておく
- 同じ人のデータが複数行あるときは、**患者ID の列** を作り、9章のように患者単位で分ける

### ⚠ 個人情報を取り除いたデータだけを使うこと

Colab にアップロードしたファイルは Google のサーバーに送られます。**患者さんのデータを、個人情報が付いたままアップロードしてはいけません。**

- 氏名・カルテ番号・生年月日・検査日などの列は削除する。患者ID は、元のカルテ番号と対応しない番号 (1, 2, 3, …) に置き換える
- ファイル名やシート名にも個人情報が残っていないか確認する
- 取り除いたデータでも、研究に使うには倫理審査や施設の許可が必要なことがある。分からないときは使う前に確認する

In [ ]:
# my = pd.read_excel('my_data.xlsx')        # CSV なら pd.read_csv('my_data.csv')
# X_my = my.drop(columns=['label'])
# y_my = my['label']
# auc = cross_val_score(xgb.XGBClassifier(random_state=0), X_my, y_my, cv=cv, scoring='roc_auc')
# print(f'AUC {auc.mean():.3f} ± {auc.std():.3f}')

## このノートブックの範囲と注意点

このノートブックは **分類器の作り方と評価の仕方** の入門です。実際の研究では、次の点に注意が必要です。

| 注意点 | 内容 |
|---|---|
| 同じ患者のデータ | 左右の臓器・複数回の検査などがあるときは、必ず患者単位で分ける (9章) |
| 答えが混じった特徴量 | 診断の **後に** 分かる情報 (治療内容など) を特徴量に入れると、性能が不当に高くなる |
| 欠測 | 「なぜ欠けているか」に意味があることがある (重症だから検査できなかった、など) |
| ラベルの偏り | 陽性が少ないと正解率は当てにならない。AUC・感度・特異度を見る |
| 外部検証 | 交差検証だけでなく、**別の施設・別の時期のデータでの検証** が必要 |
| 報告 | 予測モデルの論文では、報告ガイドライン (**TRIPOD+AI**) に沿って書くことが求められる |

### さらに学ぶための資料

- XGBoost のドキュメント — https://xgboost.readthedocs.io/
- scikit-learn の交差検証の説明 (Group 系の分け方を含む) — https://scikit-learn.org/stable/modules/cross_validation.html
- SHAP のドキュメント — https://shap.readthedocs.io/
- TRIPOD+AI：AI を使った予測モデルの報告ガイドライン (Collins et al., *BMJ*, 2024) — https://doi.org/10.1136/bmj-2023-078378

## まとめ

- **決定木** は質問を繰り返して分ける分かりやすい手法だが、1本では性能に限界があり、深くすると過学習する
- **ランダムフォレスト** は独立に作った木の多数決、**XGBoost** は間違いを直す木を順に足していく手法
- XGBoost は **表形式のデータ** の分類に強く、欠測があってもそのまま学習できる
- XGBoost の利点は、**U字型・しきい値・組み合わせ** のような関係を、形を知らなくてもデータから見つけられること。データが多いほど力を発揮する
- 関係が単純なデータでは、ロジスティック回帰と差がないこともよくある。まず単純な手法で基準を作り、複雑な手法と比べる
- 同じ患者のデータが複数行あるときは、**患者単位で分けて** 評価する
- 重要度・SHAP・部分依存プロットで、モデルが何を手がかりにしたかを確かめる

### 課題
1. 4章の決定木の図を読んで、最初の質問に使われた特徴量と、その値を確かめよう。
2. 8章で、ロジスティック回帰の学習データに `bmi` の2乗の列を加えると (`X_tr.assign(bmi2=(X_tr['bmi'] - 23) ** 2)`、評価用も同様に)、AUC はどう変わるか調べてみよう。
3. 8章の `make_risk_data` の `logit` から U字型・しきい値・組み合わせの項を消して一方向の関係だけにすると、手法ごとの差はどうなるか調べてみよう。
4. 9章で、1人あたりの行数 (`make_grouped_data(n_rows=...)`) を 2 → 5 に増やすと、行単位と患者単位の AUC の差はどう変わるか調べてみよう。
5. 10章の SHAP の図で、上位の特徴量はどちら向きに効いているか読み取ってみよう。